# actas-cnn — entrenamiento portable (Colab/Kaggle, GPU)

Notebook para entrenar **ResNet-18 estilo CIFAR** sobre el dataset
empaquetado de actas Presidenciales ONPE en una GPU gratuita (T4 en
Colab, T4×2 en Kaggle). Soporta ablations (label smoothing,
RandAugment, mixup, cosine LR).

## Antes de correr (una sola vez)

1. **Hugging Face token**: https://huggingface.co/settings/tokens →
   "Create new token" → tipo **Write** → copiar.
2. **Crear 2 repos en HF**:
   - Dataset: https://huggingface.co/new-dataset (e.g.
     `<tu-usuario>/actas-cnn-dataset`).
   - Model: https://huggingface.co/new (e.g.
     `<tu-usuario>/actas-cnn-model`).
3. **Subir `data_bundle.tar.gz`** (~460 MB) al dataset repo via la web
   UI o `huggingface-cli upload`.
4. **Actualizar `config.py`** en tu repo con `hf_dataset_repo` y
   `hf_model_repo` reales.
5. **Push del repo a GitHub** (publico) y editar `REPO_URL` abajo.
6. **Configurar HF_TOKEN en Colab Secrets** (icono de llave a la
   izquierda → "+" → name=`HF_TOKEN`, value=tu token).

Detalle completo en `docs/08-setup-colab.md`.

In [ ]:
# 1. Clonar repo + cwd
import os, sys

REPO_URL = "https://github.com/f3r21/actas-cnn.git"
REPO_DIR = "actas-cnn"

if not os.path.isdir(REPO_DIR):
    os.system(f"git clone {REPO_URL}")
os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

In [ ]:
# 2. Instalar dependencias (torch ya viene en Colab/Kaggle con CUDA)
os.system("pip install -q -r requirements.txt")

In [ ]:
# 3. Cargar HF_TOKEN desde Colab Secrets o Kaggle Secrets
def load_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

for key in ("HF_TOKEN", "WANDB_API_KEY", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY"):
    val = load_secret(key)
    if val:
        os.environ[key] = val
        print(f"  cargado {key}")
assert os.environ.get("HF_TOKEN"), "HF_TOKEN no esta configurado en Secrets"

In [ ]:
# 4. Descargar bundle y descomprimir
from huggingface_hub import hf_hub_download
from config import REMOTE
from pathlib import Path

bundle_local = hf_hub_download(
    repo_id=REMOTE.hf_dataset_repo,
    repo_type="dataset",
    filename="data_bundle.tar.gz",
)
print(f"bundle bajado: {bundle_local}")

# Descomprimir en la raiz del repo (extrae a data/, templates.json, fiducial_anchors.json)
os.system(f"tar -xzf {bundle_local} -C .")
print("descomprimido. Estructura:")
os.system("ls -lh data/ | head; echo '---'; ls data/crops_train/ | head -3")

In [ ]:
# 5. Entrenar (ablation principal)
ARCH = "resnet18"
EPOCHS = 40   # mas epochs aprovechando GPU; cosine LR se ajusta automatico
LABEL_SMOOTHING = 0.1
RANDAUGMENT = True
MIXUP = 0.2
COSINE_LR = True
SUFFIX = "colab_ls_ra_mu_cos_e40"

flags = []
if LABEL_SMOOTHING > 0:
    flags.append(f"--label-smoothing {LABEL_SMOOTHING}")
if RANDAUGMENT:
    flags.append("--randaugment")
if MIXUP > 0:
    flags.append(f"--mixup {MIXUP}")
if COSINE_LR:
    flags.append("--cosine-lr")
flags.append(f"--suffix {SUFFIX}")

cmd = (f"python train.py --manifest data/manifest_train.csv "
       f"--root data/crops_train --arch {ARCH} --epochs {EPOCHS} {' '.join(flags)}")
print(cmd)
os.system(cmd)

In [ ]:
# 6. Evaluar (digit / field / acta-level + reconstruccion totales)
ckpt = f"checkpoints/{ARCH}_{SUFFIX}_best.pt"
os.system(f"python scripts/evaluate.py --split val --checkpoint {ckpt} "
          f"--out-csv data/evaluate_val_{SUFFIX}.csv")

In [ ]:
# 7. Subir checkpoint + CSV de evaluate al HF model repo
import storage

storage.upload(ckpt, Path(ckpt).name, kind="model")
storage.upload(f"data/evaluate_val_{SUFFIX}.csv",
               f"evaluate_val_{SUFFIX}.csv", kind="model")
print("subido al HF model repo:", REMOTE.hf_model_repo)

In [ ]:
os.system("python train.py --manifest data/manifest.csv --root data/crops --arch deep --epochs 20 --push")